In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
import neurokit2 as nk
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Simulate a clean physiological ECG signal (Heart Rate = 70 bpm)
sampling_rate = 500
ecg_signal = nk.ecg_simulate(duration=5, sampling_rate=sampling_rate, heart_rate=70)

# 2. Clean the signal
ecg_cleaned = nk.ecg_clean(ecg_signal, sampling_rate=sampling_rate, method="neurokit")

# 3. Find R-peaks
_, rpeaks = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)

# 4. Delineate other waves (P, Q, S, T) using the Continuous Wavelet Transform (CWT) method
signals, waves = nk.ecg_delineate(ecg_cleaned, rpeaks, sampling_rate=sampling_rate, method="cwt")

# Time array
t = np.linspace(0, len(ecg_cleaned)/sampling_rate, len(ecg_cleaned))

In [3]:
fig_del = go.Figure()

# Plot the cleaned ECG signal
fig_del.add_trace(go.Scatter(x=t, y=ecg_cleaned, mode='lines', name='ECG Signal (Lead V5)', line=dict(color='#38bdf8', width=2)))

# Helper function to plot markers
def add_marker(wave_indices, color, symbol, name):
    # Filter out NaNs
    valid_idx = [int(i) for i in wave_indices if not np.isnan(i)]
    if len(valid_idx) > 0:
        fig_del.add_trace(go.Scatter(
            x=t[valid_idx], y=ecg_cleaned[valid_idx],
            mode='markers', marker=dict(color=color, size=10, symbol=symbol, line=dict(width=1, color='white')),
            name=name
        ))

# R-Peaks
add_marker(rpeaks['ECG_R_Peaks'], '#f43f5e', 'star', 'R-Peaks')

# P-Waves
add_marker(waves['ECG_P_Onsets'], '#a855f7', 'triangle-right', 'P-Onset')
add_marker(waves['ECG_P_Peaks'], '#a855f7', 'circle', 'P-Peak')
add_marker(waves['ECG_P_Offsets'], '#a855f7', 'triangle-left', 'P-Offset')

# T-Waves
add_marker(waves['ECG_T_Onsets'], '#10b981', 'triangle-right', 'T-Onset')
add_marker(waves['ECG_T_Peaks'], '#10b981', 'circle', 'T-Peak')
add_marker(waves['ECG_T_Offsets'], '#10b981', 'triangle-left', 'T-Offset')

fig_del.update_layout(
    title="NeuroKit2 Automated Feature Delineation (CWT Algorithm)",
    xaxis_title="Time (seconds)",
    yaxis_title="Amplitude (mV)",
    template="plotly_dark",
    height=450, margin=dict(l=20, r=20, t=40, b=20),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_del.show()

In [4]:
#| label: clinical-ludb-counts
from pathlib import Path
from collections import Counter
import wfdb
import pandas as pd
import numpy as np

ROOT = Path("..")
ludb = ROOT / "data/ludb"
records = sorted(int(p.stem) for p in ludb.glob("*.hea"))

rows = []
for rid in records:
    header = wfdb.rdheader(str(ludb / str(rid)))
    for lead in header.sig_name:
        ann = wfdb.rdann(str(ludb / str(rid)), lead)
        counts = Counter(ann.symbol)
        rows.append({
            "record": rid, "lead": lead, "p_peak": counts["p"],
            "qrs_peak": counts["N"], "t_peak": counts["t"],
            "onset": counts["("], "offset": counts[")"],
        })
ludb_counts = pd.DataFrame(rows)
ludb_counts[["p_peak", "qrs_peak", "t_peak", "onset", "offset"]].sum()

p_peak      16797
qrs_peak    21965
t_peak      19661
onset       58321
offset      58423
dtype: int64

In [5]:
#| label: clinical-isp-widths
import ast

def read_isp(split):
    frame = pd.read_csv(
        ROOT / "data/isp/isp_delineation_dataset"
        / f"{split}_isp_delineation_data.csv"
    )
    frame["intervals"] = frame.target.map(ast.literal_eval)
    return frame

isp = {"train": read_isp("train"), "test": read_isp("test")}
width_rows = []
for split, frame in isp.items():
    for intervals in frame.intervals:
        for cls, onset, offset in intervals:
            width_rows.append({
                "split": split, "class": cls, "width_samples": offset - onset
            })
pd.DataFrame(width_rows).groupby(["split", "class"]).width_samples.agg(
    ["count", "median", "mean", "std"]
)

count  median        mean        std
split class                                      
test  0        793   114.0  114.114754  19.106710
      1        900   110.0  112.154444  15.242699
      2        865   182.0  183.995376  32.847677
train 0       4312   116.0  115.485158  18.955753
      1       5048   110.0  112.829635  15.352018
      2       4788   182.0  183.285505  38.219400

In [6]:
#| label: clinical-master-summary
from pathlib import Path
import pandas as pd

ROOT = Path("..")
clinical = pd.read_csv(
    ROOT / "results/comprehensive_latest_48_models/all_48_models_master.csv"
)

fields = [
    "ptbxl_qrs_correlation", "ptbxl_st_correlation",
    "ptbxl_jpoint_mae", "ptbxl_rpeak_timing_mae_ms",
    "ecgfounder_150_macro_auroc", "ecgfounder_150_ece",
    "ptbxl_superclass_macro_auroc"
]
clinical.groupby("family")[fields].median()

,ptbxl_qrs_correlation,ptbxl_st_correlation,ptbxl_jpoint_mae,ptbxl_rpeak_timing_mae_ms,ecgfounder_150_macro_auroc,ecgfounder_150_ece,ptbxl_superclass_macro_auroc
family,,,,,,,
ecg_aim,0.939217,0.822082,0.039642,73.949863,0.879469,0.043593,0.879070
multiscale_vae,0.931759,0.806745,0.046679,78.777842,0.879470,0.045245,0.877280
unet,0.904629,0.775747,0.058932,66.928942,0.865016,0.049033,0.850736
